In [4]:
from sklearn.datasets import make_classification
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
from sklearn.model_selection import train_test_split
import torch
import torch.optim as optim

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [5]:
import pandas as pd
data=pd.read_csv("/content/drive/MyDrive/fmnist_small.csv")

In [6]:
data.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [7]:
torch.manual_seed(100)

In [9]:
X=data.iloc[:,1:].values
y=data.iloc[:,0].values

In [19]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.25,random_state=100)

In [20]:
X_train=X_train/225.0
X_test=X_test/225.0

In [47]:
class CustomDataset(Dataset):
  def __init__(self,features,labels):
    self.features=torch.tensor(features,dtype=torch.float32)
    self.labels=torch.tensor(labels,dtype=torch.long)

  def __len__(self):
    return self.features.shape[0]

  def __getitem__(self, index):
    return self.features[index],self.labels[index]

In [48]:
train_dataset=CustomDataset(X_train,y_train)

In [49]:
test_dataset=CustomDataset(X_test,y_test)

In [50]:
train_dataloader=DataLoader(dataset=train_dataset,batch_size=32,shuffle=True)
test_dataloader=DataLoader(dataset=test_dataset,batch_size=32,shuffle=False)

In [51]:
train_dataloader

In [52]:
class MyNeuralNetwork(nn.Module):
  def __init__(self,num_features):
    super().__init__()
    self.model=nn.Sequential(
        nn.Linear(num_features,128),
        nn.ReLU(),
        nn.Linear(128,64),
        nn.ReLU(),
        nn.Linear(64,10)
    )

  def forward(self,features):
    return self.model(features)

In [53]:
epochs=100
lr=0.1

In [54]:
model=MyNeuralNetwork(X_train.shape[1])
print(model)
# Cross Entropy loss already has Softmax implimented in it therefore we did not make a softmax layer in our model
criterion=nn.CrossEntropyLoss()
optimizer=optim.SGD(lr=lr,params=model.parameters())

MyNeuralNetwork(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)


In [56]:
for epoch in range(epochs):
  total_epoch_loss=0
  for batch_features,batch_labels in train_dataloader:
    y_pred=model(batch_features)
    loss=criterion(y_pred,batch_labels)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    total_epoch_loss+=loss.item()
  avg_epochl_loss=total_epoch_loss/len(train_dataloader)
  print(f"epoch : {epoch+1} , loss: {avg_epochl_loss}")

epoch : 1 , loss: 0.783584770370037
epoch : 2 , loss: 0.66462390452412
epoch : 3 , loss: 0.6118836591007016
epoch : 4 , loss: 0.545815240406821
epoch : 5 , loss: 0.5026788305728993
epoch : 6 , loss: 0.47823681972973736
epoch : 7 , loss: 0.45087626925174223
epoch : 8 , loss: 0.4304134065800525
epoch : 9 , loss: 0.41894079385497046
epoch : 10 , loss: 0.38060330499148537
epoch : 11 , loss: 0.3653318375243363
epoch : 12 , loss: 0.34767777789780435
epoch : 13 , loss: 0.3406800774606407
epoch : 14 , loss: 0.33243021828697084
epoch : 15 , loss: 0.3454701348293758
epoch : 16 , loss: 0.2935301133609833
epoch : 17 , loss: 0.2961025136582395
epoch : 18 , loss: 0.2828155349913641
epoch : 19 , loss: 0.27941666717859026
epoch : 20 , loss: 0.25443808837456905
epoch : 21 , loss: 0.2591488849081046
epoch : 22 , loss: 0.24299141019582748
epoch : 23 , loss: 0.24058680888925885
epoch : 24 , loss: 0.22494453821886093
epoch : 25 , loss: 0.2082045522141964
epoch : 26 , loss: 0.21449095124365591
epoch : 27 , 

In [59]:
total=0
correct=0
with torch.no_grad():
  for batch_features,batch_labels in test_dataloader:
    y_pred=model(batch_features)
    _,predicted=torch.max(y_pred,1)
    total+=batch_labels.shape[0]
    correct+=(predicted==batch_labels).sum().item()
  print(correct/total)

0.8453333333333334
